# AGILAB API benchmark in Colab

This notebook demonstrates AGILAB benchmark mode from Google Colab.
It clones the AGILAB repository, installs it in editable mode, and
benchmarks `mycode_project` across two local execution modes:

- `AGI.PYTHON_MODE`
- `AGI.CYTHON_MODE`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThalesGroup/agilab/blob/main/examples/notebook_quickstart/agi_core_colab_benchmark.ipynb)


In [ ]:
!if [ -d /content/agilab/.git ]; then cd /content/agilab && git pull --ff-only; else rm -rf /content/agilab && git clone --depth 1 https://github.com/ThalesGroup/agilab.git /content/agilab; fi
!python -m pip uninstall -y agilab agi-cluster agi-node agi-env >/dev/null 2>&1 || true
!python -m pip install -q uv
!pip install -q -e /content/agilab/src/agilab/core/agi-env -e /content/agilab/src/agilab/core/agi-node -e /content/agilab/src/agilab/core/agi-cluster -e /content/agilab


In [ ]:
import os
import pathlib
import subprocess
from io import UnsupportedOperation as IOUnsupportedOperation
from pathlib import Path
import sys
import json
os.environ["IS_SOURCE_ENV"] = "1"
os.environ.pop("IS_WORKER_ENV", None)

if not hasattr(pathlib, "UnsupportedOperation"):
    pathlib.UnsupportedOperation = IOUnsupportedOperation

for name in [module for module in list(sys.modules) if module == "agi_cluster" or module.startswith("agi_cluster.") or module == "agi_env" or module.startswith("agi_env.") or module == "agi_node" or module.startswith("agi_node.")]:
    del sys.modules[name]
import importlib
importlib.invalidate_caches()

REPO_ROOT = Path("/content/agilab")
CORE_NODE_SRC = REPO_ROOT / "src" / "agilab" / "core" / "agi-node" / "src"
CORE_CLUSTER_SRC = REPO_ROOT / "src" / "agilab" / "core" / "agi-cluster" / "src"
CORE_ENV_SRC = REPO_ROOT / "src" / "agilab" / "core" / "agi-env" / "src"
LOCAL_CORE_PATHS = [REPO_ROOT / "src", CORE_NODE_SRC, CORE_ENV_SRC, CORE_CLUSTER_SRC]
os.environ["PYTHONPATH"] = os.pathsep.join(
    [str(path) for path in LOCAL_CORE_PATHS]
    + ([os.environ["PYTHONPATH"]] if os.environ.get("PYTHONPATH") else [])
)
for path in LOCAL_CORE_PATHS:
    sys.path.insert(0, str(path))

import types
agi_cluster_pkg = types.ModuleType("agi_cluster")
agi_cluster_pkg.__path__ = [str(CORE_CLUSTER_SRC / "agi_cluster")]
sys.modules["agi_cluster"] = agi_cluster_pkg
import agi_node

from agi_cluster.agi_distributor import AGI
from agi_env import AgiEnv

APP = "mycode_project"
APPS_PATH = REPO_ROOT / "src" / "agilab" / "apps"
CORE_PACKAGES = [
    REPO_ROOT / "src" / "agilab" / "core" / "agi-env",
    REPO_ROOT / "src" / "agilab" / "core" / "agi-node",
    REPO_ROOT / "src" / "agilab" / "core" / "agi-cluster",
]
_core_env_ready = set()

def ensure_app_core_packages(app_root: Path) -> None:
    if app_root in _core_env_ready:
        return
    cmd = [
        "uv",
        "--preview-features",
        "extra-build-dependencies",
        "pip",
        "install",
        "--project",
        str(app_root),
        "-e",
        str(CORE_PACKAGES[0]),
        "-e",
        str(CORE_PACKAGES[1]),
        "-e",
        str(CORE_PACKAGES[2]),
    ]
    subprocess.run(cmd, check=True)
    _core_env_ready.add(app_root)

ensure_app_core_packages(APPS_PATH / APP)
app_env = AgiEnv(apps_path=APPS_PATH, app=APP, verbose=0)
print("Repo root:", REPO_ROOT)
print("Apps path:", APPS_PATH)
print("Benchmark file:", app_env.benchmark)


In [ ]:
benchmark_json = await AGI.run(
    app_env,
    scheduler="127.0.0.1",
    workers={"127.0.0.1": 1},
    mode=[AGI.PYTHON_MODE, AGI.CYTHON_MODE],
)
benchmark_json
